In [1]:
import pandas as pd
import warnings

warnings.simplefilter(action="ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 500)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

In [2]:
df = pd.read_csv("persona.csv")

In [3]:
# Quick summary of the DataFrame
def quick_summary(dataframe, head=5):
    print("\n##################### Shape #####################")
    print(dataframe.shape)
    print("\n##################### Types #####################")
    print(dataframe.dtypes)
    print("\n##################### NA #####################")
    print(dataframe.isnull().sum())
    print("##################### Head #####################")
    print(dataframe.head(head))
    print("\n##################### Tail #####################")
    print(dataframe.tail(head))
    print("\n##################### Quantiles #####################")
    print(dataframe.describe([0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T)


quick_summary(df)


##################### Shape #####################
(5000, 5)

##################### Types #####################
PRICE       int64
SOURCE     object
SEX        object
COUNTRY    object
AGE         int64
dtype: object

##################### NA #####################
PRICE      0
SOURCE     0
SEX        0
COUNTRY    0
AGE        0
dtype: int64
##################### Head #####################
   PRICE   SOURCE   SEX COUNTRY  AGE
0     39  android  male     bra   17
1     39  android  male     bra   17
2     49  android  male     bra   17
3     29  android  male     tur   17
4     49  android  male     tur   17

##################### Tail #####################
      PRICE   SOURCE     SEX COUNTRY  AGE
4995     29  android  female     bra   31
4996     29  android  female     bra   31
4997     29  android  female     bra   31
4998     39  android  female     bra   31
4999     29  android  female     bra   31

##################### Quantiles #####################
        count  mean   std   mi

In [4]:
df["SOURCE"].nunique()

2

In [5]:
df["SOURCE"].value_counts()

SOURCE
android    2974
ios        2026
Name: count, dtype: int64

In [6]:
df["PRICE"].nunique()

6

In [7]:
df["PRICE"].value_counts()

PRICE
29    1305
39    1260
49    1031
19     992
59     212
9      200
Name: count, dtype: int64

In [8]:
df["COUNTRY"].nunique()

6

In [9]:
df["COUNTRY"].value_counts()

COUNTRY
usa    2065
bra    1496
deu     455
tur     451
fra     303
can     230
Name: count, dtype: int64

In [10]:
df.groupby("COUNTRY").agg({"PRICE": "count"}).sort_values(by="PRICE", ascending=False)

,PRICE
COUNTRY,
usa,2065
bra,1496
deu,455
tur,451
fra,303
can,230


In [11]:
df.pivot_table(values="PRICE", index="COUNTRY", aggfunc="count")

,PRICE
COUNTRY,
bra,1496
can,230
deu,455
fra,303
tur,451
usa,2065


In [12]:
df.groupby("COUNTRY").agg({"PRICE": "sum"}).sort_values("PRICE", ascending=False)

,PRICE
COUNTRY,
usa,70225
bra,51354
tur,15689
deu,15485
fra,10177
can,7730


In [13]:
df.groupby("COUNTRY").agg({"PRICE": "mean"}).sort_values(by="PRICE", ascending=False)

,PRICE
COUNTRY,
tur,34.79
bra,34.33
deu,34.03
usa,34.01
can,33.61
fra,33.59


In [14]:
df.groupby("SOURCE").agg({"PRICE": "mean"}).sort_values(by="PRICE", ascending=False)

,PRICE
SOURCE,
android,34.17
ios,34.07


In [15]:
df.groupby(["COUNTRY", "SOURCE"]).agg({"PRICE": "mean"}).sort_values(
    by="PRICE", ascending=False
)

,,PRICE
COUNTRY,SOURCE,
tur,android,36.23
bra,android,34.39
usa,ios,34.37
fra,android,34.31
deu,ios,34.27
bra,ios,34.22
can,ios,33.95
deu,android,33.87
usa,android,33.76


In [16]:
agg_df = (
    df.groupby(["COUNTRY", "SOURCE", "SEX", "AGE"])
    .agg({"PRICE": "mean"})
    .sort_values(by="PRICE", ascending=False)
)

agg_df.head()

,,,,PRICE
COUNTRY,SOURCE,SEX,AGE,
bra,android,male,46,59.00
usa,android,male,36,59.00
fra,android,female,24,59.00
usa,ios,male,32,54.00
deu,android,female,36,49.00


In [17]:
agg_df.reset_index(inplace=True)
agg_df.head()

,COUNTRY,SOURCE,SEX,AGE,PRICE
0,bra,android,male,46,59.00
1,usa,android,male,36,59.00
2,fra,android,female,24,59.00
3,usa,ios,male,32,54.00
4,deu,android,female,36,49.00


In [18]:
bins = [0, 18, 23, 30, 40, agg_df["AGE"].max()]
labels = ["0_18", "19_23", "24_30", "31_40", "41_" + str(agg_df["AGE"].max())]

agg_df["AGE_CAT"] = pd.cut(agg_df["AGE"], bins=bins, labels=labels)

agg_df.head()

,COUNTRY,SOURCE,SEX,AGE,PRICE,AGE_CAT
0,bra,android,male,46,59.00,41_66
1,usa,android,male,36,59.00,31_40
2,fra,android,female,24,59.00,24_30
3,usa,ios,male,32,54.00,31_40
4,deu,android,female,36,49.00,31_40


In [19]:
agg_df["customers_level_based"] = pd.concat(
    [
        agg_df["COUNTRY"].str.upper(),
        agg_df["SOURCE"].str.upper(),
        agg_df["SEX"].str.upper(),
        agg_df["AGE_CAT"].astype(str),
    ],
    axis=1,
).agg("_".join, axis=1)

In [20]:
agg_df = agg_df[["customers_level_based", "PRICE"]]
agg_df.head()

,customers_level_based,PRICE
0,BRA_ANDROID_MALE_41_66,59.00
1,USA_ANDROID_MALE_31_40,59.00
2,FRA_ANDROID_FEMALE_24_30,59.00
3,USA_IOS_MALE_31_40,54.00
4,DEU_ANDROID_FEMALE_31_40,49.00


In [21]:
agg_df["SEGMENT"] = pd.qcut(agg_df["PRICE"], 4, labels=["D", "C", "B", "A"])

segment_summary = agg_df.groupby("SEGMENT", observed=True).agg(
    {"PRICE": ["mean", "max", "sum"]}
)

In [22]:
new_user_1 = "TUR_ANDROID_FEMALE_31_40"
new_user_2 = "FRA_IOS_FEMALE_31_40"

result_user_1 = agg_df[agg_df["customers_level_based"] == new_user_1]
result_user_2 = agg_df[agg_df["customers_level_based"] == new_user_2]

results = pd.concat([result_user_1, result_user_2], axis=0).reset_index(drop=True)

total_expected_revenue = results["PRICE"].sum()
results.loc["TOTAL"] = ["", total_expected_revenue, ""]

print(results)

          customers_level_based  PRICE SEGMENT
0      TUR_ANDROID_FEMALE_31_40  43.00       A
1      TUR_ANDROID_FEMALE_31_40  40.67       A
2          FRA_IOS_FEMALE_31_40  33.00       C
3          FRA_IOS_FEMALE_31_40  32.64       C
TOTAL                           149.30        
